# 03 — Random Forest

**Purpose:** Train the supervised multiclass classifier (5 classes).  
**Acceptance criterion:** macro F1 ≥ 0.80 on ≥ 3 of 4 attack categories, measured on **val** set.  
If criterion not met, iterate on features (return to extractor tasks 1.x) before touching model.

**Input:** `training/splits/train.parquet`, `training/splits/val.parquet`  
**Output:** `training/models/rf_vN.pkl`, `training/results/rf_val_report.txt`

**Rule R2:** Hyperparameter tuning uses val only. Test set is locked until notebook 05.

## 1. Load Features

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SPLITS  = Path("../../training/splits")
RESULTS = Path("../../training/results")
MODELS  = Path("../../training/models")
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(SPLITS / "train.parquet")
val_df   = pd.read_parquet(SPLITS / "val.parquet")

DROP_COLS = [
    "status_code", "req_count_1s", "req_count_5s", "req_count_60s",
    "error_rate_4xx_60s", "endpoint_diversity_60s", "_source",
    "sample_id", "timestamp", "_row_hash",
]
train_df.drop(columns=[c for c in DROP_COLS if c in train_df.columns], inplace=True)
val_df.drop(columns=[c for c in DROP_COLS if c in val_df.columns], inplace=True)

y_train = train_df.pop("label")
y_val   = val_df.pop("label")
X_train, X_val = train_df, val_df

TARGET_NAMES = ["benign", "cmdi", "path_traversal", "sqli", "xss"]

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(y_train.value_counts())

Train: (265104, 66)  Val: (56809, 66)
label
sqli              159138
benign             69379
xss                20912
path_traversal     11787
cmdi                3888
Name: count, dtype: int64


## 2. Hyperparameter Search

In [2]:
from sklearn.ensemble import RandomForestClassifier

# Fixed config from the F4.4 memory investigation (docs/results.md):
# n=30, max_depth=25 is the smallest configuration that reaches 4/4
# attack classes >= 0.80 F1 while staying under the ONNX memory gate.
# SMOTE confirmed negative result (docs/limitations.md #1) - not applied.
best_rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)

print(f"Params: {best_rf.get_params()}")


Params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced_subsample', 'criterion': 'gini', 'max_depth': 25, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 30, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## 3. Best Model Evaluation on Val

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay

y_pred_rf = best_rf.predict(X_val)

print("=== Random Forest — Validation Set ===")
print(classification_report(y_val, y_pred_rf, target_names=TARGET_NAMES))

macro_f1  = f1_score(y_val, y_pred_rf, average="macro")
per_class = f1_score(y_val, y_pred_rf, average=None,
                     labels=["cmdi", "path_traversal", "sqli", "xss"])

print(f"Macro F1           : {macro_f1:.4f}")
print(f"Attack classes >= 0.80: {sum(f >= 0.80 for f in per_class)}/4")

assert sum(f >= 0.80 for f in per_class) >= 3, (
    "GATE FAILED: F1 >= 0.80 in fewer than 3 attack classes. "
    "Return to feature engineering (tasks 1.x) before proceeding."
)

=== Random Forest — Validation Set ===


                precision    recall  f1-score   support

        benign       1.00      1.00      1.00     14867
          cmdi       0.90      0.93      0.91       833
path_traversal       0.97      0.98      0.97      2525
          sqli       1.00      1.00      1.00     34103
           xss       1.00      0.98      0.99      4481

      accuracy                           0.99     56809
     macro avg       0.97      0.98      0.97     56809
  weighted avg       0.99      0.99      0.99     56809



Macro F1           : 0.9740
Attack classes >= 0.80: 4/4


## 4. Feature Importance

In [4]:
importances = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)

print("Top 20 features (mean impurity decrease):")
print(importances.head(20).to_string())

low_importance = importances[importances < 0.001].index.tolist()
print(f"\nFeatures with importance < 0.001 : {len(low_importance)}")
print("Candidates for pruning:", low_importance)

# Permutation importance (more reliable — use on val, not train)
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_rf, X_val, y_val,
    n_repeats=10, random_state=42, n_jobs=-1, scoring="f1_macro",
)
perm_series = pd.Series(
    perm.importances_mean, index=X_train.columns
).sort_values(ascending=False)

print("\nTop 20 features (permutation importance on val):")
print(perm_series.head(20).to_string())

Top 20 features (mean impurity decrease):
path_length                 0.076317
ua_length                   0.071735
special_char_ratio          0.046013
traversal_sequence_count    0.045018
payload_entropy             0.041753
uri_length                  0.040674
sqli_operator_count         0.039611
encoded_char_freq           0.036048
path_depth                  0.036032
sqli_keyword_density        0.034688
path_separator_count        0.033532
alert_function_present      0.032716
numeric_char_ratio          0.031464
xss_marker_density          0.030217
xss_marker_count            0.029229
url_encoded_ratio           0.027664
body_entropy                0.027536
query_string_length         0.027468
html_tag_count              0.025211
uppercase_ratio             0.024279



Features with importance < 0.001 : 15
Candidates for pruning: ['pipe_count', 'sensitive_extension_count', 'dollar_sign_count', 'subshell_count', 'ua_suspicious', 'newline_char_count', 'extended_ascii_ratio', 'fragment_present', 'backtick_count', 'javascript_url_count', 'inline_style_present', 'unicode_escape_count', 'authorization_length', 'null_byte_count', 'dotdot_encoded_count']



Top 20 features (permutation importance on val):
path_length                 0.071096
ua_length                   0.060479
body_entropy                0.025232
query_param_count           0.023644
special_char_ratio          0.022942
uppercase_ratio             0.021539
path_depth                  0.020842
uri_length                  0.019105
content_type_encoded        0.016708
body_length                 0.014576
numeric_char_ratio          0.013537
command_separator_count     0.010433
encoded_char_freq           0.010095
sqli_keyword_density        0.009871
shell_command_count         0.009690
traversal_sequence_count    0.007913
query_string_length         0.007315
path_separator_count        0.006723
payload_entropy             0.006548
url_encoded_ratio           0.004520


## 5. Save Model

In [5]:
import joblib

model_path = MODELS / "rf_v6.pkl"
joblib.dump(best_rf, model_path)
print(f"Model saved : {model_path}")
print(f"Params      : {best_rf.get_params()}")

Model saved : ../../training/models/rf_v6.pkl
Params      : {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced_subsample', 'criterion': 'gini', 'max_depth': 25, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 30, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


## 6. Confusion Matrix

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred_rf,
    display_labels=TARGET_NAMES,
    ax=ax,
    colorbar=False,
    xticks_rotation=30,
)
plt.title("Random Forest — Confusion Matrix (val set)")
plt.tight_layout()

fig_path = RESULTS / "rf_confusion_matrix.png"
plt.savefig(fig_path, dpi=150)
plt.close()
print(f"Saved to {fig_path}")

Saved to ../../training/results/rf_confusion_matrix.png
